In [27]:
from abc import ABC
from torch.utils.data import Dataset, IterableDataset, DataLoader
import pickle
import torch
import pytorch_lightning as pl
import random
import gc
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import Dataset, IterableDataset, DataLoader
import os
import torch
import numpy as np
from sklearn.preprocessing import PowerTransformer, QuantileTransformer, RobustScaler
from dataclasses import dataclass, field
from typing import List,Optional,Set


@dataclass
class Batch:
    """
    A batch of data, with non-optional x, y, and target_y attributes.
    All other attributes are optional.

    If you want to add an attribute for testing only, you can just assign it after creation like:
    ```
        batch = Batch(x=x, y=y, target_y=target_y)
        batch.test_attribute = test_attribute
    ```
    """
    # Required entries
    x: torch.Tensor
    y: torch.Tensor
    target_y: torch.Tensor
    model_names: List[str] = field(default_factory=list)

    # Optional Batch Entries
    style: Optional[torch.Tensor] = None
    style_hyperparameter_values: Optional[torch.Tensor] = None
    single_eval_pos: Optional[torch.Tensor] = None
    causal_model_dag: Optional[object] = None
    mean_prediction: Optional[bool] = None  # this controls whether to do mean prediction in bar_distribution for nonmyopic BO

    def other_filled_attributes(self, set_of_attributes: Set[str] = frozenset(('x', 'y', 'target_y'))):
        return [f.name for f in fields(self)
                if f.name not in set_of_attributes and
                getattr(self, f.name) is not None]


def load_pickle(file_path):
    with open(file_path, 'rb') as handle:
        inst = pickle.load(handle)
    return inst

classification_datasets = ['Amazon_employee_access', 'anneal', 'APSFailure', 'bank-marketing', 'Bank_Customer_Churn', 'Bioresponse', 'blood-transfusion-service-center', 'churn', 'coil2000_insurance_policies', 'credit-g', 'credit_card_clients_default', 'customer_satisfaction_in_airline', 'diabetes', 'E-CommereShippingData', 'Fitness_Club', 'GiveMeSomeCredit', 'hazelnut-spread-contaminant-detection', 'heloc', 'hiva_agnostic', 'HR_Analytics_Job_Change_of_Data_Scientists',
 'in_vehicle_coupon_recommendation', 'Is-this-a-good-customer',  'Marketing_Campaign', 'maternal_health_risk', 'MIC', 'NATICUSdroid', 'online_shoppers_intention', 'polish_companies_bankruptcy', 'qsar-biodeg', 'SDSS17', 'seismic-bumps', 'splice', 'students_dropout_and_academic_success', 'taiwanese_bankruptcy_prediction', 'website_phishing', 'jm1']

def torch_nanmean(x, axis=0, return_nanshare=False):
    num = torch.where(torch.isnan(x), torch.full_like(x, 0), torch.full_like(x, 1)).sum(axis=axis)
    value = torch.where(torch.isnan(x), torch.full_like(x, 0), x).sum(axis=axis)
    if return_nanshare:
        return value / num, 1. - num / x.shape[axis]
    return value / num

def torch_nanstd(x, axis=0):
    num = torch.where(torch.isnan(x), torch.full_like(x, 0), torch.full_like(x, 1)).sum(axis=axis)
    value = torch.where(torch.isnan(x), torch.full_like(x, 0), x).sum(axis=axis)
    mean = value / num
    mean_broadcast = torch.repeat_interleave(mean.unsqueeze(axis), x.shape[axis], dim=axis)
    return torch.sqrt(torch.nansum(torch.square(mean_broadcast - x), axis=axis) / (num - 1))


def normalize_data(data, normalize_positions=-1, return_scaling=False):
    if normalize_positions > 0:
        mean = torch_nanmean(data[:normalize_positions], axis=0)
        std = torch_nanstd(data[:normalize_positions], axis=0) + .000001
    else:
        mean = torch_nanmean(data, axis=0)
        std = torch_nanstd(data, axis=0) + .000001
    data = (data - mean) / std
    data = torch.clip(data, min=-100, max=100)

    if return_scaling:
        return data, (mean, std)
    return data


def feature_subsampling(x: torch.Tensor, num_feature: int, max_feature_dim: int) -> torch.Tensor:
    # Randomly choose feature indices without replacement
    idx = torch.randperm(num_feature)[:max_feature_dim]
    # Sort to keep feature order
    idx, _ = torch.sort(idx)
    return x[:, idx]


def feature_scale(x: torch.Tensor, 
                  num_feature: int, 
                  max_feature_dim: int,
                  rescale_with_sqrt: bool = False) -> torch.Tensor:
    scale = num_feature / max_feature_dim
    if rescale_with_sqrt:
        scale = scale ** 0.5
    return x / scale


def pfn_transform(eval_xs,
                  max_feature_dim):
    """
    :param eval_xs: the inputs
    :param preprocess_transform: str, 'none', 'power', 'quantile', 'robust'
    :param eval_position: train-x <--eval_position---> test_x
    :param normalize_with_test: when perform (x-mean)/std, whether to include test_x
    :param rescale_with_sqrt: when rescale the features, whether to use "* sqrt(num-max-feat/num-used-feat)"
    """
    num_feature = eval_xs.shape[-1]
    if num_feature > max_feature_dim:
        eval_xs = feature_subsampling(x=eval_xs, num_feature=num_feature,max_feature_dim = max_feature_dim)
    eval_xs = normalize_data(eval_xs, normalize_positions=-1)
    # Rescale X
    eval_xs = feature_scale(x=eval_xs, num_feature=eval_xs.shape[-1], rescale_with_sqrt=False, max_feature_dim=max_feature_dim)
    if eval_xs.shape[-1] < max_feature_dim:
        eval_xs = torch.cat([eval_xs,
                            torch.zeros((eval_xs.shape[0], max_feature_dim - num_feature), device=eval_xs.device)],
                             dim=-1)
    return eval_xs



def sample(df: pd.DataFrame, 
           category_column: str,
           anomaly_ratio: float,
           random_state: int = None,
           norm_rank: int = 1,
           max_normal_sample=5000) -> torch.Tensor:
    counts = df[category_column].value_counts()
    norm_rank = min(len(counts), norm_rank)
    normal_class = counts.index[norm_rank - 1]

    normal_df = df[df[category_column] == normal_class].copy()
    normal_df['anomaly_label'] = 0
    n_norm = min(len(normal_df),max_normal_sample)
    normal_df = normal_df.sample(n=n_norm, random_state=random_state).copy()
    other_df = df[df[category_column] != normal_class].copy()

    
    if anomaly_ratio >= 1.0:
        raise ValueError("anomaly_ratio must be less than 1")
    n_anom = int(anomaly_ratio * n_norm / (1.0 - anomaly_ratio))
    n_anom = min(n_anom, len(other_df))

    anomaly_df = other_df.sample(n=n_anom, random_state=random_state).copy()
    anomaly_df['anomaly_label'] = 1

    combined = pd.concat([normal_df, anomaly_df]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    combined = combined.drop(columns=[category_column])

    # Keep only numeric features; ensure label is last column
    y = combined['anomaly_label'].astype(np.float32).to_numpy()
    X = combined.drop(columns=['anomaly_label']).select_dtypes(include=[np.number]).astype(np.float32).to_numpy()
    out = np.hstack([X, y.reshape(-1, 1)])
    return torch.from_numpy(out)


classification_datasets = ['Amazon_employee_access', 'anneal', 'APSFailure', 'bank-marketing', 'Bank_Customer_Churn', 'Bioresponse', 'blood-transfusion-service-center', 'churn', 'coil2000_insurance_policies', 'credit-g', 'credit_card_clients_default', 'customer_satisfaction_in_airline', 'diabetes', 'E-CommereShippingData', 'Fitness_Club', 'GiveMeSomeCredit', 'hazelnut-spread-contaminant-detection', 'heloc', 'hiva_agnostic', 'HR_Analytics_Job_Change_of_Data_Scientists',
 'in_vehicle_coupon_recommendation', 'Is-this-a-good-customer',  'Marketing_Campaign', 'maternal_health_risk', 'MIC', 'NATICUSdroid', 'online_shoppers_intention', 'polish_companies_bankruptcy', 'qsar-biodeg', 'SDSS17', 'seismic-bumps', 'splice', 'students_dropout_and_academic_success', 'taiwanese_bankruptcy_prediction', 'website_phishing', 'jm1']



class ValidationDataset(Dataset):
    def __init__(
        self,
        max_feature_dim: int = 100,
        anomaly_ratio: float = 0.4,
    ):
        self.classification_datasets = classification_datasets
        self.max_feature_dim = max_feature_dim
        self.anomaly_ratio = anomaly_ratio

        # metadata: dataset_name -> target_feature
        metadata_path = '../data/TabArena/metadata/tabarena_dataset_metadata.csv'
        meta_df = pd.read_csv(metadata_path)
        
        self.name_to_tgt = dict(zip(meta_df["dataset_name"], meta_df["target_feature"]))
        self.data_root = '../data/TabArena/datasets/'
        
        # dataset classes counts
        self.dataset_dict = {}
        self.dataset_list = []  # List to store dataset names for indexing
        self.num_episodes = 0
        for i in classification_datasets:
            data_name = i
            data_path = self.data_root + f'/{data_name}.csv'
            df = pd.read_csv(data_path)
            name_to_tgt_feature_dict = dict(zip(meta_df['dataset_name'], meta_df['target_feature']))
            tgt_fea = name_to_tgt_feature_dict[data_name]
            counts = df[tgt_fea].value_counts()
            self.dataset_dict[data_name] = len(counts)
            for j in range(len(counts)):
                tgt = self.name_to_tgt[data_name]
                batch = sample(df, category_column=tgt, anomaly_ratio=self.anomaly_ratio,
                       random_state=42, norm_rank=j)
                # Split features/labels
                X = batch[:, :-1]
                y = batch[:, -1].long()
                if X.shape[0] > 1000:
                    self.dataset_list.append(f"{data_name}%{j}")
            self.num_episodes += len(counts)
    
    def set_rank(self, rank):
        self.rank = rank
            

    def __len__(self):
        return self.num_episodes

    def __getitem__(self, idx):
        # Allow referencing by index
        #print(self.dataset_list)
        dataset_name_rank = self.dataset_list[idx]
        dataset_name = dataset_name_rank.split('%')[0]
        rank = int(dataset_name_rank.split('%')[1])
            
        data_path = os.path.join(self.data_root, f"{dataset_name}.csv")
        df = pd.read_csv(data_path)
        tgt = self.name_to_tgt[dataset_name]

        # Your sample() should return a torch.Tensor with label in the last column
        batch = sample(df, category_column=tgt, anomaly_ratio=self.anomaly_ratio,
                       random_state=42, norm_rank=rank)
        # Split features/labels
        X = batch[:, :-1]
        y = batch[:, -1].long()

        # Transform + pad/truncate features to max_feature_dim
        X = pfn_transform(eval_xs=X, max_feature_dim=self.max_feature_dim)  # -> (N, max_feature_dim)
        return {"X": X, "y": y}  # X: (N, D), y: (N,)
    
    def prior_batch_collate_fn(self, batch_list):
        if len(batch_list) != 1:
            raise ValueError("ValidationDataset should only return one item per batch.")
        
        # Extract features and labels
        xs = batch_list[0]['X']
        ys = batch_list[0]['y']
        
        # Separate inliers and outliers
        inliers = xs[ys == 0]
        outliers = xs[ys == 1]
        
        # seq_len: total inliers 
        # single_eval_pos: train inliers
        # num_test_x: test inliers
        seq_len = inliers.shape[0]
        single_eval_pos = int(seq_len * 0.5)
        num_inliers = single_eval_pos
        num_test_x = seq_len - single_eval_pos
        
        train_inliers= inliers[:single_eval_pos]
        test_inliers = inliers[single_eval_pos:]
        test_la = outliers
        test_x = torch.cat([test_inliers, test_la], dim=0)
        test_y = torch.tensor([0] * num_test_x + [1] * test_la.shape[0])
        
        x = torch.cat([train_inliers, test_x], dim=0)
        x = x.reshape(1, x.shape[0], x.shape[1]).to(torch.float32)
        y = torch.cat([torch.tensor([-33] * num_inliers), test_y], dim = 0)
        y = y.reshape(1, y.shape[0]).to(torch.float32)
        
        return Batch(
            x=x.transpose(0, 1), 
            y=None, 
            target_y=y.transpose(0, 1), 
            model_names=None, 
            single_eval_pos=single_eval_pos
        )






In [28]:
val = ValidationDataset()
for idx,i in enumerate(val):
    print(idx, val.dataset_list[idx])
    X = i['X']
    y = i['y']
    print(X.shape, y.shape)

0 Amazon_employee_access%0
torch.Size([3161, 100]) torch.Size([3161])
1 Amazon_employee_access%1
torch.Size([6897, 100]) torch.Size([6897])
2 APSFailure%0
torch.Size([2291, 100]) torch.Size([2291])
3 APSFailure%1
torch.Size([6375, 100]) torch.Size([6375])
4 bank-marketing%0
torch.Size([8333, 100]) torch.Size([8333])
5 bank-marketing%1
torch.Size([8333, 100]) torch.Size([8333])
6 Bank_Customer_Churn%0
torch.Size([3395, 100]) torch.Size([3395])
7 Bank_Customer_Churn%1
torch.Size([7037, 100]) torch.Size([7037])
8 Bioresponse%0
torch.Size([2861, 100]) torch.Size([2861])
9 Bioresponse%1
torch.Size([3390, 100]) torch.Size([3390])
10 churn%0
torch.Size([1178, 100]) torch.Size([1178])
11 churn%1
torch.Size([5000, 100]) torch.Size([5000])
12 coil2000_insurance_policies%1
torch.Size([5586, 100]) torch.Size([5586])
13 credit_card_clients_default%0
torch.Size([8333, 100]) torch.Size([8333])
14 credit_card_clients_default%1
torch.Size([8333, 100]) torch.Size([8333])
15 customer_satisfaction_in_airl

In [33]:
something = val[1]

In [34]:
batch= val.prior_batch_collate_fn([something])
batch.target_y
batch.x.shape, batch.target_y.shape

(torch.Size([6897, 1, 100]), torch.Size([6897, 1]))

In [35]:
batch.single_eval_pos

2500

In [36]:
batch.target_y[batch.single_eval_pos:]
print(batch.target_y[batch.single_eval_pos:].sum() / len(batch.target_y[batch.single_eval_pos:]))

tensor(0.4314)


In [ ]:

import pandas as pd

def sample(df: pd.DataFrame, category_column: str, anomaly_ratio: float, random_state: int = None, norm_rank: int=1) -> pd.DataFrame:
    """
    Sample a dataset into 'normal' and 'anomaly' subsets based on a categorical column.

    - The most frequent category is treated as the normal class.
    - Anomalies are randomly sampled from all other classes according to the specified ratio.

    Parameters:
    - df: Input DataFrame containing the data.
    - category_column: Name of the categorical column used to define classes.
    - anomaly_ratio: Fraction of normal samples to include as anomalies (e.g., 0.1 for 10%).
    - random_state: Optional seed for reproducibility.
    - norm_rank: choose the top-norm_rank frequent class as norm samples


    Returns:
    - A DataFrame consisting of all normal samples and a random subset of anomalies.
    """
    # Identify the normal class as the most frequent category
    counts = df[category_column].value_counts()
    # normal_class = counts.idxmax() # use the top 1
    norm_rank = min(len(counts), norm_rank)
    normal_class = counts.index[norm_rank - 1]

    # Split normals and potential anomalies
    normal_df = df[df[category_column] == normal_class].copy()
    normal_df['anomaly_label'] = [0] * len(normal_df)
    other_df = df[df[category_column] != normal_class]

    # Determine number of anomalies to sample
    n_norm = len(normal_df)
    if anomaly_ratio >= 1.0:
        raise ValueError("anomaly_ratio must be less than 1")
    n_anom = int(anomaly_ratio * n_norm / (1.0 - anomaly_ratio))
    n_anom = min(n_anom, len(other_df))  # 不超过 available


    # Sample anomalies
    anomaly_df = other_df.sample(n=n_anom, random_state=random_state)
    anomaly_df['anomaly_label'] = [1] * len(anomaly_df)

    # Combine and shuffle
    combined = pd.concat([normal_df, anomaly_df]).sample(frac=1, random_state=random_state).reset_index(drop=True)

    combined = combined.drop(category_column, axis=1)

    return combined


tabarena_classification_datasets = ['Amazon_employee_access', 'anneal', 'APSFailure', 'bank-marketing', 'Bank_Customer_Churn', 'Bioresponse', 'blood-transfusion-service-center', 'churn', 'coil2000_insurance_policies', 'credit-g', 'credit_card_clients_default', 'customer_satisfaction_in_airline', 'diabetes', 'Diabetes130US', 'E-CommereShippingData', 'Fitness_Club', 'GiveMeSomeCredit', 'hazelnut-spread-contaminant-detection', 'heloc', 'hiva_agnostic', 'HR_Analytics_Job_Change_of_Data_Scientists',
 'in_vehicle_coupon_recommendation', 'Is-this-a-good-customer', 'kddcup09_appetency', 'Marketing_Campaign', 'maternal_health_risk', 'MIC', 'NATICUSdroid', 'online_shoppers_intention', 'polish_companies_bankruptcy', 'qsar-biodeg', 'SDSS17', 'seismic-bumps', 'splice', 'students_dropout_and_academic_success', 'taiwanese_bankruptcy_prediction', 'website_phishing', 'jm1']

# Example usage:
if __name__ == "__main__":

    if 0: print(len(tabarena_classification_datasets))

    if 1: # tablib full data
        data_name = 'table_809642_809642.csv'
        metadata_path = '/data/haominwe/code/repurpose_real_data/data/TabLib/metadata/metadata.csv'
        meta_df = pd.read_csv(metadata_path)
        name_to_tgt_feature_dict = dict(zip(meta_df['dataset_name'], meta_df['target_feature']))

        data_file = '/data/haominwe/code/repurpose_real_data/data/TabLib/datasets/'
        data_path = data_file + f'/{data_name}'
        df = pd.read_csv(data_path)

        tgt_fea = name_to_tgt_feature_dict[data_name]
        sampled_df = sample(df, category_column=tgt_fea, anomaly_ratio=0.2, random_state=42, norm_rank=1)
        print(sampled_df)

        pass

    if 0: # tablib small data
        data_name = 'table_014441_014441.csv'
        metadata_path = '/data/haominwe/code/repurpose_real_data/data/TabLib/metadata/metadata_small.csv'
        meta_df = pd.read_csv(metadata_path)
        name_to_tgt_feature_dict = dict(zip(meta_df['dataset_name'], meta_df['target_feature']))

        data_file = '/data/haominwe/code/repurpose_real_data/data/TabLib/tablib_csv_en_gt1000_v5/'
        data_path = data_file + f'/{data_name}'
        df = pd.read_csv(data_path)

        tgt_fea = name_to_tgt_feature_dict[data_name]
        sampled_df = sample(df, category_column=tgt_fea, anomaly_ratio=0.2, random_state=42, norm_rank=1)
        print(sampled_df)

    if 0: # dataset in tabzilla
        data_name = 'audiology'
        metadata_path = './data/TabZilla/metadata/tabzilla_dataset_metadata.csv'
        meta_df = pd.read_csv(metadata_path)
        name_to_tgt_feature_dict = dict(zip(meta_df['dataset_name'], meta_df['target_feature']))

        data_file = '/data/haominwe/code/repurpose_real_data/data/TabZilla/datasets/'
        data_path = data_file + f'/{data_name}.csv'
        df = pd.read_csv(data_path)

        tgt_fea = name_to_tgt_feature_dict[data_name]
        sampled_df = sample(df, category_column=tgt_fea, anomaly_ratio=0.2, random_state=42, norm_rank=1)
        print(sampled_df)



    if 0:  # dataset in Tabarena
        data_name = 'anneal' # choose one from classification_datasets
        metadata_path = '/data/haominwe/code/repurpose_real_data/data/TabArena/metadata/tabarena_dataset_metadata.csv'
        meta_df = pd.read_csv(metadata_path)
        name_to_tgt_feature_dict = dict(zip(meta_df['dataset_name'], meta_df['target_feature']))

        data_file = '/data/haominwe/code/repurpose_real_data/data/TabArena/datasets/'
        data_path = data_file + f'/{data_name}.csv'
        df = pd.read_csv(data_path)

        tgt_fea = name_to_tgt_feature_dict[data_name]
        sampled_df = sample(df, category_column=tgt_fea, anomaly_ratio=0.2, random_state=42, norm_rank=1)
        # in sampled df: anomaly_label=0 means norm points, anomaly_label=1 means anomaly
        # norm_rank: we first sort all class by frequency, then set the top-norm_rank class as norm samples
        print(sampled_df)


    if 0: # test function for sample
        # Create a sample DataFrame
        df = pd.DataFrame({
            'feature1': range(100),
            'category': ['A'] * 80 + ['B'] * 15 + ['C'] * 5
        })
        sampled_df = sample(df, 'category', anomaly_ratio=0.2, random_state=42)
        # print(sampled_df['category'].value_counts())

        print(sampled_df)
        print('Anomaly ratio:', sampled_df['anomaly_label'].mean())



